In [2]:

# @title Install dependencies
!pip -q install langchain langchain-community langchain-google-genai google-generativeai \
               faiss-cpu pypdf cohere langchain-experimental reportlab


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==

In [3]:
!pip -q install transformers sentencepiece accelerate


In [4]:

# @title Set API keys
import os, getpass

if "GEMINI_API_KEY" not in os.environ or not os.environ["GEMINI_API_KEY"]:
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter GEMINI_API_KEY: ")

if "COHERE" not in os.environ or not os.environ["COHERE"]:
    os.environ["COHERE"] = getpass.getpass("Enter COHERE: ")

print("Environment variables set.")


Enter GEMINI_API_KEY: ··········
Enter COHERE: ··········
Environment variables set.


In [5]:
# @title Imports & basic config (patched)
from typing import List, Dict, Any, Optional
import os, json, re, math

from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain.prompts import PromptTemplate

import google.generativeai as genai
import cohere


if not os.environ.get("GEMINI_API_KEY"):
    raise ValueError("GEMINI_API_KEY is not set. Run the earlier cell to input it.")
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

GEMINI_MODEL = "gemini-2.5-flash"
EMBED_MODEL  = "models/text-embedding-004"
# COHERE_RERANK_MODEL = "rerank-english-v3.0"
COHERE_RERANK_MODEL = "rerank-v3.5"



llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    temperature=0.2,
    convert_system_message_to_human=True,
    google_api_key=os.environ["GEMINI_API_KEY"],  # <-- explicit key here
)

embeddings = GoogleGenerativeAIEmbeddings(
    model=EMBED_MODEL,
    google_api_key=os.environ["GEMINI_API_KEY"],  # <-- explicit key here too
)


if not os.environ.get("COHERE"):
    raise ValueError("COHERE key is not set. Run the earlier cell to input it.")
co = cohere.Client(os.environ["COHERE"])

print("LLM + Embeddings + Cohere configured.")


LLM + Embeddings + Cohere configured.


In [6]:

# @title Select PDF paths

DATASHEET_PDF = "/content/Datasheet.pdf"
EVENTCODES_PDF = "/content/EventCodes.pdf"


import os
for f in [DATASHEET_PDF, EVENTCODES_PDF]:
    print(f, "OK" if os.path.exists(f) else "MISSING - upload it or change the path")


/content/Datasheet.pdf OK
/content/EventCodes.pdf OK


In [7]:

# @title Load PDFs into Documents
def load_pdf(path: str) -> List[Document]:
    loader = PyPDFLoader(path)
    return loader.load()

datasheet_docs = load_pdf(DATASHEET_PDF)
event_docs = load_pdf(EVENTCODES_PDF)

print(f"Loaded {len(datasheet_docs)} pages from Datasheet")
print(f"Loaded {len(event_docs)} pages from Event Codes")


Loaded 2 pages from Datasheet
Loaded 21 pages from Event Codes


In [8]:

# @title Semantic-aware chunking

chunker = SemanticChunker(embeddings, breakpoint_threshold_type="percentile", breakpoint_threshold_amount=0.4)

datasheet_chunks = chunker.split_documents(datasheet_docs)
eventcode_chunks = chunker.split_documents(event_docs)

print(f"Datasheet chunks: {len(datasheet_chunks)}")
print(f"Event codes chunks: {len(eventcode_chunks)}")

# test
def preview(doc: Document, n=250):
    txt = doc.page_content.strip().replace("\n", " ")
    return txt[:n] + ("..." if len(txt) > n else "")
print("Example datasheet chunk:", preview(datasheet_chunks[0]))
print("Example event codes chunk:", preview(eventcode_chunks[0]))


Datasheet chunks: 45
Event codes chunks: 159
Example datasheet chunk: www.victronenergy.com     Victron Energy B.V.
Example event codes chunk: Event Codes.


In [9]:

# @title Create FAISS stores and retrievers
error_vs = FAISS.from_documents(eventcode_chunks, embeddings)
data_vs = FAISS.from_documents(datasheet_chunks, embeddings)


error_code_retriever = error_vs.as_retriever(search_kwargs={"k": 16})
datasheet_retriever = data_vs.as_retriever(search_kwargs={"k": 16})

def cohere_rerank(query: str, docs: List[Document], top_n: int = 6) -> List[Document]:
    if not docs:
        return []
    res = co.rerank(
        model=COHERE_RERANK_MODEL,
        query=query,
        documents=[d.page_content for d in docs],
        top_n=min(top_n, len(docs)),
    )

    ranked = [docs[r.index] for r in res.results]
    return ranked

def retrieve_with_rerank(retriever, query: str, initial_k: int = 16, top_n: int = 6) -> List[Document]:
    base_docs = retriever.get_relevant_documents(query)
    return cohere_rerank(query, base_docs, top_n=top_n)

print("Retrievers ready.")


Retrievers ready.


In [10]:

# @title Build the classifier
router_prompt = PromptTemplate(
    input_variables=["query"],
    template=(
        "You are a routing classifier. Decide how to handle the user's query.\n"
        "Rules:\n"
        " - If the query is related to error, code, or fault: route to 'error_codes'\n"
        " - If the query is related to features or specifications: route to 'datasheet'\n"
        " - If the query requests the current AC power of an inverter: route to 'tool'\n"
        " - Otherwise: 'ambiguous'\n\n"
        "If route is 'tool', extract the inverter_id if present (like IS_1654).\n"
        "Return valid compact JSON with keys: route, inverter_id (string or null), reason.\n\n"
        "User query: {query}\n"
        "JSON:"
    )
)

def classify_query(query: str) -> Dict[str, Any]:
    msg = router_prompt.format(query=query)
    resp = llm.invoke(msg)
    txt = resp.content
    try:
        data = json.loads(txt)
    except Exception:

        m = re.search(r"\{.*\}", txt, flags=re.S)
        data = json.loads(m.group(0)) if m else {"route": "ambiguous", "inverter_id": None, "reason": "Parse failure"}

    data.setdefault("route", "ambiguous")
    data.setdefault("inverter_id", None)
    data.setdefault("reason", "")
    return data

print(classify_query("Show me fault 501 meaning"))
print(classify_query("What is the AC output voltage spec?"))
print(classify_query("What is current AC power of inverter IS_1655?"))


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


{'route': 'error_codes', 'inverter_id': None, 'reason': 'The query is related to a fault.'}


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


{'route': 'datasheet', 'inverter_id': None, 'reason': 'The query is related to specifications (AC output voltage spec).'}


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


{'route': 'tool', 'inverter_id': 'IS_1655', 'reason': "The query explicitly asks for the 'current AC power of an inverter'."}


In [11]:

# @title Define the inverter AC power function (tool)
def get_inverter_ac_power(inverter_id: str) -> str:

    if inverter_id == "IS_1654":
        return "100.0"
    elif inverter_id == "IS_1655":
        return "200.0"
    elif inverter_id == "IS_1656":
        return "300.0"
    elif inverter_id == "IS_1657":
        return "400.0"
    elif inverter_id == "IS_1658":
        return "500.0"
    else:
        return "Unknown inverter ID"

print("Power tool test:", get_inverter_ac_power("IS_1655"))


Power tool test: 200.0


In [12]:

# @title Answer synthesis helpers
answer_prompt = PromptTemplate(
    input_variables=["question", "snippets"],
    template=(
        "You are a helpful inverter support assistant.\n"
        "Use the provided context snippets to answer the user's question accurately.\n"
        "Cite **source & page** numbers inline like [source:page] when using specific facts.\n"
        "If insufficient info, say so briefly.\n\n"
        "Question:\n{question}\n\n"
        "Context snippets:\n{snippets}\n\n"
        "Answer:"
    )
)

def format_snippets(docs: List[Document]) -> str:
    lines = []
    for i, d in enumerate(docs, 1):
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", "N/A")
        preview = d.page_content.strip().replace("\n", " ")
        if len(preview) > 600:
            preview = preview[:600] + "..."
        lines.append(f"[{i}] (source={src}, page={page}) {preview}")
    return "\n".join(lines)

def generate_answer(question: str, docs: List[Document]) -> str:
    if not docs:
        return "I couldn't retrieve relevant context for this question."
    snippets = format_snippets(docs)
    msg = answer_prompt.format(question=question, snippets=snippets)
    resp = llm.invoke(msg)
    return resp.content


In [13]:
# @title Agentic RAG pipeline


def agentic_answer(query: str, top_n: int = 6) -> Dict[str, Any]:


    try:
        route_info = classify_query(query)
        if not isinstance(route_info, dict):
            raise ValueError(f"classify_query returned non-dict: {type(route_info)}")
    except Exception as e:
        return {
            "route": "ambiguous",
            "inverter_id": None,
            "answer": f"Routing error: {e}. Try a more explicit question (fault/spec/tool).",
            "docs": []
        }

    route = route_info.get("route", "ambiguous")
    inverter_id = route_info.get("inverter_id")

    outcome = {"route": route, "inverter_id": inverter_id, "answer": "", "docs": []}

    if route == "tool":
        if not inverter_id or inverter_id == "null":
            outcome["answer"] = "Please provide a valid inverter_id (e.g., IS_1654, IS_1655)."
            return outcome
        try:
            power_w = get_inverter_ac_power(inverter_id)
            outcome["answer"] = f"Current AC power for inverter {inverter_id}: {power_w} W"
        except Exception as e:
            outcome["answer"] = f"Tool error: {e}"
        return outcome

    elif route == "error_codes":
        try:
            docs = retrieve_with_rerank(error_code_retriever, query, top_n=top_n)
            outcome["docs"] = docs
            outcome["answer"] = generate_answer(query, docs)
        except Exception as e:
            outcome["answer"] = f"Retrieval/rerank error (error_codes): {e}"
        return outcome

    elif route == "datasheet":
        try:
            docs = retrieve_with_rerank(datasheet_retriever, query, top_n=top_n)
            outcome["docs"] = docs
            outcome["answer"] = generate_answer(query, docs)
        except Exception as e:
            outcome["answer"] = f"Retrieval/rerank error (datasheet): {e}"
        return outcome

    else:
        outcome["answer"] = (
            "Your request is a bit unclear. Are you asking about an **error/fault code**, "
            "**specification/feature**, or the **current AC power** of a specific inverter?"
        )
        return outcome

# Smoke tests
print(agentic_answer("What is fault 501?")["route"])
print(agentic_answer("What is the output voltage specification?")["route"])
print(agentic_answer("Get current AC power of inverter IS_1658")["route"])


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
/tmp/ipython-input-4190669437.py:23: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  base_docs = retriever.get_relevant_documents(query)
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


error_codes


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


datasheet


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


tool


In [14]:

# @title Ask a question
user_q = input("Enter your question: ").strip()
result = agentic_answer(user_q, top_n=6)

print("Route:", result["route"], "| Inverter ID:", result.get("inverter_id"))
print("\n=== ANSWER ===\n", result["answer"])

if result.get("docs"):
    print("\n=== TOP CONTEXT ===")
    for i, d in enumerate(result["docs"], 1):
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", "N/A")
        print(f"[{i}] source={src} page={page}")


Enter your question: What is the issue related to 502?


/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:357: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


Route: error_codes | Inverter ID: None

=== ANSWER ===
 The issue related to event 502 is a "Grid incident" where the grid frequency is not within the permissible range, causing the inverter to disconnect from the utility grid [source:4].

=== TOP CONTEXT ===
[1] source=/content/EventCodes.pdf page=4
[2] source=/content/EventCodes.pdf page=5
[3] source=/content/EventCodes.pdf page=6
[4] source=/content/EventCodes.pdf page=2
[5] source=/content/EventCodes.pdf page=3
[6] source=/content/EventCodes.pdf page=3
